# Exploração dos dados

Nesta primeira etápa, é essêncial buscar por possíveis tendências e entendimento de como cada feature da base funciona.

In [ ]:
#Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

In [ ]:
# Salvando o banco
bd = pd.read_csv('dados_clientes.csv')
bd

In [ ]:
# Entendendo as variáveis
bd.info()

Para compreender melhor, quais são os diferenciais entre cada um dos tipos de usuário, seria interessante separá-los em dois dataframes e analisar cada um separadamente.

In [ ]:
df_noChurn = bd[bd['churn'] == 0]
df_noChurn.info()

In [ ]:
df_churn = bd[bd['churn'] == 1]
df_churn.info()

In [ ]:
df_noChurn.describe(include='all')

In [ ]:
df_churn.describe(include='all')

Após olhar de forma superficial os dados, foi possível concluir:
- Não existe valores nulos nos dados fornecidos.
- Algumas idades apresentaram incônsistencias, apresentando valores negativos.
- Alguns casos apareceram sem nenhum valor gasto.
- Existem mais homens no plano do que mulheres e mais mulheres cancelaram do que homens.
- A média de idade mostra que a maioria dos usuários são adultos.
- Mais pessoas casadas cancelaram o plano.
- Existem caractéres inválidos na coluna `'produtos_assinados'`.

# Pré-processamento dos dados
Com base no que foi identificado na exploração dos dados, agora é o momento de corrigir os erros encontrados e deixarem os dados "limpos" para o trabalho.


## Corrigindo erros encontrados na exploração

In [ ]:
# Corrigindo as idades negativas, mudando elas para a média

# Calculando a média das idades válidas
media_idades = bd.loc[bd['idade'] >= 0, 'idade'].mean()

# Substituindo as idades negativas pela média
bd.loc[bd['idade'] < 0, 'idade'] = media_idades

print(bd['idade'].min())


In [ ]:
# Corrigindo valores = 0 para a média

# Calculando a média dos valores mensais válidos (diferentes de zero)
media_valor_mensal = bd.loc[bd['valor_mensal'] != 0, 'valor_mensal'].mean()

# Substituindo os valores mensais iguais a zero pela média
bd.loc[bd['valor_mensal'] == 0, 'valor_mensal'] = media_valor_mensal

# Calculando a média dos valores totais gastos válidos (diferentes de zero)
media_valor_total = bd.loc[bd['total_gasto'] != 0, 'total_gasto'].mean()

# Substituindo os valores totais gastos iguais a zero pela média
bd.loc[bd['total_gasto'] == 0, 'total_gasto'] = media_valor_total

print(bd['valor_mensal'].min())
print(bd['total_gasto'].min())


In [ ]:
# Substituindo valores inválidos por outro valor aleatório da coluna

# Encontrando as linhas com 'produtos_assinados' vazio
empty_products_rows = bd[bd['produtos_assinados'] == '[]'].index

# Selecionando um valor aleatório da coluna 'produtos_assinados' (excluindo os vazios)
non_empty_products = bd.loc[bd['produtos_assinados'] != '[]', 'produtos_assinados'].sample(n=1).iloc[0]

# Preenchendo as linhas vazias com o valor aleatório
bd.loc[empty_products_rows, 'produtos_assinados'] = non_empty_products

bd['produtos_assinados'].value_counts()


## Buscando outliers

Após explorar os dados, foram encontradas duas colunas com grande variação, sendo elas: `'valor_mensal', 'total_gasto'`. Essa diferença nos dados pode enviesar no momento do treinamento do modelo, logo a melhor saida seria tratar esses outliers.

In [ ]:
# Normalizando o valor_mensal

# Calculando o IQR para 'valor_mensal'
Q1_valor_mensal = bd['valor_mensal'].quantile(0.25)
Q3_valor_mensal = bd['valor_mensal'].quantile(0.75)
IQR_valor_mensal = Q3_valor_mensal - Q1_valor_mensal

# Definindo os limites superior e inferior para outliers
limite_superior_valor_mensal = Q3_valor_mensal + 1.5 * IQR_valor_mensal
limite_inferior_valor_mensal = Q1_valor_mensal - 1.5 * IQR_valor_mensal

# Substituindo os outliers pelo limite superior ou inferior
bd['valor_mensal'] = np.where(bd['valor_mensal'] > limite_superior_valor_mensal, limite_superior_valor_mensal, bd['valor_mensal'])
bd['valor_mensal'] = np.where(bd['valor_mensal'] < limite_inferior_valor_mensal, limite_inferior_valor_mensal, bd['valor_mensal'])

bd.describe()

In [ ]:
# Normalizando o total_gasto

# Calculando o IQR para 'total_gasto'
Q1_total_gasto = bd['total_gasto'].quantile(0.25)
Q3_total_gasto = bd['total_gasto'].quantile(0.75)
IQR_total_gasto = Q3_total_gasto - Q1_total_gasto

# Definindo os limites superior e inferior para outliers
limite_superior_total_gasto = Q3_total_gasto + 1.5 * IQR_total_gasto
limite_inferior_total_gasto = Q1_total_gasto - 1.5 * IQR_total_gasto

# Substituindo os outliers pelo limite superior ou inferior
bd['total_gasto'] = np.where(bd['total_gasto'] > limite_superior_total_gasto, limite_superior_total_gasto, bd['total_gasto'])
bd['total_gasto'] = np.where(bd['total_gasto'] < limite_inferior_total_gasto, limite_inferior_total_gasto, bd['total_gasto'])

bd.describe()


## Categorização

Esta etápa é essêncial para o funcionamento do modelo, já que com varíaveis não numéricas é impossível executa-lo. Sendo assim, a ideia desta subseção é agrupar as variáveis categóricas e transforma-las em numéricas.

In [ ]:
categorical_columns = bd.select_dtypes(include=['object']).columns

print("Variáveis categóricas identificadas:")
print(categorical_columns)

In [ ]:

# Codificando a coluna 'genero'
bd['genero'] = bd['genero'].map({'Masculino': 0, 'Feminino': 1})
bd


In [ ]:
# Codificando o estado civil
bd['estado_civil'] = bd['estado_civil'].map({'Solteiro': 0, 'Casado': 1, 'Divorciado': 2})
bd

In [ ]:
# Codificando o tipo de contrato
bd['tipo_contrato'] = bd['tipo_contrato'].map({'Mensal': 0, 'Anual': 1})
bd

In [ ]:
# Codificando forma de pagamento

bd['forma_pagamento'] = bd['forma_pagamento'].map({'Boleto': 0, 'Cartão': 1, 'Débito': 2})
bd


In [ ]:
# Codificando a renda

bd['renda_faixa'] = bd['renda_faixa'].map({'0-1000': 0, '1000-5000': 1, '5000-10000': 2, '10000-50000': 3, '50000-100000': 4})
bd

Após esse pré-processamento, pode-se concluir que as variáveis categorizadas até este momento são o bastante para o treinamento do modelo. Existe mais uma váriavel categorica que não foi codificada, porém ela é tão varíada que não apresenta um padrão que possa concluir em um fator que influênciara no treino.

# Geração de insights e formulação de hipóteses
Após toda a limpeza nos dados e exploração dos dados, é possível tirar alguns insights que podem justificar algumas das razões na qual os usuários estao cancelando seus planos.

Para isso, a estratégia de criar hipóteses e confronta-las é utilizada.

## Hipótese 1 - A maioria dos usuários que cancelaram o plano entraram em contato com o suporte ao menos uma vez.

Essa hipotése pode mostrar que usuários que tendem a usar o suporte estão pensando em desistir dos planos, caso seja verdade, pode ser um ponto de grande atenção à empresa.

In [ ]:
df_churn['suporte_contatado'].value_counts().plot(kind='bar')
plt.title('Contagem de Contatos com Suporte para Usuários que Cancelaram')
plt.xlabel('Contato com Suporte')
plt.ylabel('Quantidade de Usuários')
plt.show()


In [ ]:
print("Usuários que contataram o suporte:", df_churn[df_churn['suporte_contatado'] != 0].shape[0])
print("Usuários que NÃO contataram o suporte:", df_churn[df_churn['suporte_contatado'] == 0].shape[0])


Como é possível ver, a hipótese é verdadeira, possuindo uma grande quantidade que chegaram a dois contatos, sendo importante uma verificação quando os usuários chegam a esse número de contatos.

## Hipótese 2 - A maioria das mulheres cancelaram o plano

No momento da exploração, foi descoberto que a maioria de casos de cancelamento veio de mulheres, partindo deste principio, é importante entender como funciona o público feminino com o plano. Caso essa hipótese seja verdade, é provável que exista um viés masculino nos produtos, o que é ruim para qualquer marca.

In [ ]:
women_no_churn = df_noChurn[df_noChurn['genero'] == "Feminino"].shape[0]
women_churn = df_churn[df_churn['genero'] == "Feminino"].shape[0]

print(f"Mulheres que não cancelaram: {women_no_churn}")
print(f"Mulheres que cancelaram: {women_churn}")
print(f"Total de mulheres: {women_no_churn + women_churn}")


Com base no resultado, a hipótese se prova falsa, mostrando que o motivo pelo cancelamento dos planos não vem de um viés baseado no genêro dos usuários.

## Hipótese 3 - Usuários que cancelam costumam ter maiores atrasos de pagamento.

In [ ]:
df_churn['atrasos_pagamento'].value_counts().plot(kind='bar')
plt.title('Contagem de Atrasos de Pagamento para Usuários que Cancelaram')
plt.xlabel('Atrasos de Pagamento')
plt.ylabel('Quantidade de Usuários')
plt.show()


df_noChurn['atrasos_pagamento'].value_counts().plot(kind='bar')
plt.title('Contagem de Atrasos de Pagamento para Usuários que Não Cancelaram')
plt.xlabel('Atrasos de Pagamento')
plt.ylabel('Quantidade de Usuários')
plt.show()


Com base nos dois gráficos, é possível entender que a hipótese é verdadeira, a maioria dos usuários que não cancelam não possuem atrasos, já a minoria dos que cancelaram não atrasaram, sendo possível concluir que a partir de um momento que os usuários começam a atrasar mais, seria importante contatar estes clientes.

# Treinamento, teste de modelos, avaliação e comparação de modelos
Nesta etápa serão desenvolvidos os modelos de IA que deverão predizer quem são os clientes que geralmente saiem.

Para realizar, serão utilizados dois algoritmos, o primeiro RandomForest e o segundo Logistic Regression. E para ambos terem uma melhor combinação de parâmetros, será utilizado o GridSearch, um algoritmo de hiperparemitrazação que usa todas as combinações possíveis e entrega a que teve melhor resultado.

In [ ]:
# Criando as varíaveis de treino

# Separando as features (X) e o target (y)
X = bd.drop(['churn', 'id_cliente', 'produtos_assinados'], axis=1)
y = bd['churn']

# Dividindo os dados em conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Realiza o ajuste de hiperparâmetros para melhorar o desempenho dos modelos

# === Tuning de hiperparâmetros para o RandomForestClassifier ===

# Define a grade de parâmetros a ser testada para o modelo Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],          # Número de árvores na floresta
    'max_depth': [None, 10, 20],             # Profundidade máxima da árvore
    'min_samples_split': [2, 5, 10],         # Número mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4]            # Número mínimo de amostras por folha
}

# Instancia o modelo de Random Forest com semente aleatória para reprodutibilidade
rf_model = RandomForestClassifier(random_state=42)

# Aplica busca em grade (grid search) com validação cruzada (5 folds)
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Treina o modelo com os dados de treino
grid_search.fit(X_train, y_train)

# Obtém o melhor modelo encontrado na busca
best_rf_model = grid_search.best_estimator_

# Realiza previsões no conjunto de teste com o melhor modelo
y_pred_rf = best_rf_model.predict(X_test)

# Avalia a acurácia do modelo ajustado
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Acurácia do modelo RandomForest ajustado: {accuracy_rf}")

# Exibe o relatório de classificação com métricas como precisão, recall e F1-score
print(classification_report(y_test, y_pred_rf))

# Exibe a matriz de confusão
print(confusion_matrix(y_test, y_pred_rf))


# === Tuning de hiperparâmetros para o modelo de Regressão Logística ===

# Define a grade de parâmetros a ser testada para o modelo Logistic Regression
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],     # Parâmetro de regularização (quanto menor, maior a regularização)
    'penalty': ['l1', 'l2'],                # Tipo de penalização usada (L1 para Lasso, L2 para Ridge)
    'solver': ['liblinear', 'saga']         # Solvers que suportam penalização L1
}

# Instancia o modelo de Regressão Logística com número máximo de iterações aumentado
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# Aplica busca em grade (grid search) com validação cruzada (5 folds)
grid_search_lr = GridSearchCV(estimator=lr_model, param_grid=param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)

# Treina o modelo com os dados de treino
grid_search_lr.fit(X_train, y_train)

# Obtém o melhor modelo encontrado na busca
best_lr_model = grid_search_lr.best_estimator_

# Realiza previsões no conjunto de teste com o melhor modelo
y_pred_lr = best_lr_model.predict(X_test)

# Avalia a acurácia do modelo ajustado
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Acurácia do modelo de Regressão Logística ajustado: {accuracy_lr}")

# Exibe o relatório de classificação com métricas como precisão, recall e F1-score
print(classification_report(y_test, y_pred_lr))

# Exibe a matriz de confusão
print(confusion_matrix(y_test, y_pred_lr))


Com esses resultados é possível concluir que, para essa aplicação, o melhor algoritmo seria a regressão logística. Apesar dos valores de acurácia estarem iguais, o recall da regressão logística nos casos de churn = 1 é maior, o que seria mais seguro para esse modelo, já que o objetivo é localizar os usuários que sairão do plano, e com um recall maior, diminui o número de falsos negativos.

# Geração do arquivo de previsões

Esta é a etápa final, onde haverá a aplicação do modelo treinado em um conjunto de dados novo.

Para um melhor fluxo, foi decidido montar um pipeline feito manualmente, criando funções que executam todos os passos anteriores, facilitando a manutenção, sendo necessário apenas modificar a função necessária.

In [ ]:
# Pipeline

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

# Pré-processamento dos dados
def preprocess_data(bd):
    media_idades = bd.loc[bd['idade'] >= 0, 'idade'].mean()
    bd.loc[bd['idade'] < 0, 'idade'] = media_idades

    media_valor_mensal = bd.loc[bd['valor_mensal'] != 0, 'valor_mensal'].mean()
    bd.loc[bd['valor_mensal'] == 0, 'valor_mensal'] = media_valor_mensal

    media_valor_total = bd.loc[bd['total_gasto'] != 0, 'total_gasto'].mean()
    bd.loc[bd['total_gasto'] == 0, 'total_gasto'] = media_valor_total

    empty_products_rows = bd[bd['produtos_assinados'] == '[]'].index
    non_empty_products = bd.loc[bd['produtos_assinados'] != '[]', 'produtos_assinados'].sample(n=1).iloc[0]
    bd.loc[empty_products_rows, 'produtos_assinados'] = non_empty_products

    Q1_valor_mensal = bd['valor_mensal'].quantile(0.25)
    Q3_valor_mensal = bd['valor_mensal'].quantile(0.75)
    IQR_valor_mensal = Q3_valor_mensal - Q1_valor_mensal
    limite_superior_valor_mensal = Q3_valor_mensal + 1.5 * IQR_valor_mensal
    limite_inferior_valor_mensal = Q1_valor_mensal - 1.5 * IQR_valor_mensal
    bd['valor_mensal'] = np.where(bd['valor_mensal'] > limite_superior_valor_mensal, limite_superior_valor_mensal, bd['valor_mensal'])
    bd['valor_mensal'] = np.where(bd['valor_mensal'] < limite_inferior_valor_mensal, limite_inferior_valor_mensal, bd['valor_mensal'])

    Q1_total_gasto = bd['total_gasto'].quantile(0.25)
    Q3_total_gasto = bd['total_gasto'].quantile(0.75)
    IQR_total_gasto = Q3_total_gasto - Q1_total_gasto
    limite_superior_total_gasto = Q3_total_gasto + 1.5 * IQR_total_gasto
    limite_inferior_total_gasto = Q1_total_gasto - 1.5 * IQR_total_gasto
    bd['total_gasto'] = np.where(bd['total_gasto'] > limite_superior_total_gasto, limite_superior_total_gasto, bd['total_gasto'])
    bd['total_gasto'] = np.where(bd['total_gasto'] < limite_inferior_total_gasto, limite_inferior_total_gasto, bd['total_gasto'])

    bd['genero'] = bd['genero'].map({'Masculino': 0, 'Feminino': 1})
    bd['estado_civil'] = bd['estado_civil'].map({'Solteiro': 0, 'Casado': 1, 'Divorciado': 2})
    bd['tipo_contrato'] = bd['tipo_contrato'].map({'Mensal': 0, 'Anual': 1})
    bd['forma_pagamento'] = bd['forma_pagamento'].map({'Boleto': 0, 'Cartão': 1, 'Débito': 2})
    bd['renda_faixa'] = bd['renda_faixa'].map({'0-1000': 0, '1000-5000': 1, '5000-10000': 2, '10000-50000': 3, '50000-100000': 4})

    return bd

# Treinamento do modelo de Regressão Logística com GridSearchCV
def train_modelLR(bd):
    X = bd.drop(['churn', 'id_cliente', 'produtos_assinados'], axis=1)
    y = bd['churn']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    param_grid_lr = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    }

    lr_model = LogisticRegression(random_state=42, max_iter=1000)
    grid_search_lr = GridSearchCV(estimator=lr_model, param_grid=param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search_lr.fit(X_train, y_train)
    best_lr_model = grid_search_lr.best_estimator_

    return best_lr_model

# Predição e exportação dos resultados para CSV
def predict_csv(model, new_bd):
    new_bd = preprocess_data(new_bd)
    new_X = new_bd.drop(['produtos_assinados'], axis=1)
    results = model.predict(new_X.drop('id_cliente', axis = 1))
    results_df = pd.DataFrame({'id_cliente': new_X['id_cliente'], 'churn': results})
    results_df.to_csv('resultado_guilherme_carvalho.csv', index=False)


In [ ]:
# Está célula exporta o csv com os resultados:
bd_train = pd.read_csv('dados_clientes.csv')
bd_test = pd.read_csv('desafio.csv')
predict_csv(train_modelLR(preprocess_data(bd_train)), bd_test)